In [ ]:
# BASICS
import pandas as pd
import numpy as np
import os
import statistics
import io

from IPython.display import display
import sklearn
from scipy.stats import t, randint

# VISUALIZATION
from matplotlib import pyplot as plt
import seaborn as sns


# DATA PREP
from sklearn.utils import shuffle
from sklearn import preprocessing

# MODELS
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

# METRICS
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_auc_score
from sklearn.metrics import roc_curve, auc, roc_auc_score, make_scorer, precision_recall_curve, recall_score
from sklearn.metrics import f1_score, precision_recall_fscore_support

In [ ]:
# Grouping the features into specified categories

SDOH_Categories = ['sdoh_PCP','sdoh_INS','sdoh_HOUSING', 'sdoh_FOOD','sdoh_UTIL','sdoh_TRANS',
                   'sdoh_EMPLOY','sdoh_CLOTH_CHILD_PHONE',
                   'sdoh_DV','sdoh_HIV','sdoh_COVID','sdoh_DIABETES','sdoh_ASTHMA',
                   'sdoh_BILL_RX_HEALTHED', 'sdoh_EMOTIONAL' , 'sdoh_ALC_SUBSTANCE_ABUSE' ,'sdoh_SAFETY'
                   ,'sdoh_HOME_EQUIP' ,'sdoh_LEGAL', 'sdoh_OTHER']

Categoricals = [ 'PCP_res', 'PCP_nores','insurance_res','insurance_nores','housing_res',

                 'housing_nores','food_res','food_nores','util_res','util_nores','trans_res','trans_nores',

                 'employ_res','employ_nores','cloth_child_phone_res','cloth_child_phone_nores',
                 'dv_res','dv_nores','hiv_res','hiv_nores','covid_res','covid_nores','diabetes_res','diabetes_nores',
                 'asthma_res','asthma_nores','bill_rx_healthed_res','bill_rx_healthed_nores','emotional_res','emotional_nores',

                 'alc_substance_res','alc_substance_nores','safety_res','safety_nores','home_equip_res',
                 'home_equip_nores','legal_res','legal_nores','other_res','other_nores',



                ]



print('SDOH Categories = ', len(SDOH_Categories))


SDOH Categories =  20


In [ ]:
# Correlation matrix for total notes
numerical_data = notes_data[SDOH_Categories].select_dtypes(include=np.number)

correlation_matrix = numerical_data.corr()

num_patients = numerical_data.shape[0]
print(f"Number of patients used for correlation: {num_patients}")

display(correlation_matrix.style.background_gradient(cmap='coolwarm'))

Number of patients used for correlation: 2620


,sdoh_PCP,sdoh_INS,sdoh_HOUSING,sdoh_FOOD,sdoh_UTIL,sdoh_TRANS,sdoh_EMPLOY,sdoh_CLOTH_CHILD_PHONE,sdoh_DV,sdoh_HIV,sdoh_COVID,sdoh_DIABETES,sdoh_ASTHMA,sdoh_BILL_RX_HEALTHED,sdoh_EMOTIONAL,sdoh_ALC_SUBSTANCE_ABUSE,sdoh_SAFETY,sdoh_HOME_EQUIP,sdoh_LEGAL,sdoh_OTHER
sdoh_PCP,1.000000,0.309311,0.125194,0.082640,0.072214,0.127429,0.050210,0.052270,-0.017019,0.113594,0.041557,-0.004725,-0.028965,0.045888,0.001962,0.049335,-0.018916,-0.030323,-0.010392,0.010564
sdoh_INS,0.309311,1.000000,-0.015833,0.062555,0.073777,0.007929,0.088689,0.029258,0.039155,0.148029,0.011649,-0.010309,-0.003412,0.178286,-0.041574,0.048653,0.003767,-0.055854,-0.009038,0.011133
sdoh_HOUSING,0.125194,-0.015833,1.000000,0.073198,0.024912,0.025865,0.028965,0.137681,-0.002313,-0.014267,-0.003569,-0.041069,-0.034024,-0.044300,0.224297,-0.001456,0.225701,-0.055615,-0.008983,0.020553
sdoh_FOOD,0.082640,0.062555,0.073198,1.000000,0.209967,0.127500,-0.016543,-0.018020,0.122822,0.006955,0.093853,0.034519,0.093687,0.023489,0.143927,0.078273,0.115860,-0.091056,0.045180,0.014569
sdoh_UTIL,0.072214,0.073777,0.024912,0.209967,1.000000,0.217627,0.015166,0.041526,0.066374,-0.016115,0.025805,0.134336,0.018418,0.091163,0.081740,0.012517,0.096484,0.029170,-0.012845,0.001934
sdoh_TRANS,0.127429,0.007929,0.025865,0.127500,0.217627,1.000000,0.052708,0.093023,0.072739,0.007810,0.038700,0.045566,0.039055,0.007576,-0.007503,-0.025612,0.020734,0.046970,-0.014920,0.029519
sdoh_EMPLOY,0.050210,0.088689,0.028965,-0.016543,0.015166,0.052708,1.000000,0.039117,0.056812,0.089178,-0.027796,0.068219,0.024884,0.059327,-0.059281,0.015391,-0.058895,-0.040004,-0.009628,-0.026329
sdoh_CLOTH_CHILD_PHONE,0.052270,0.029258,0.137681,-0.018020,0.041526,0.093023,0.039117,1.000000,nan,-0.005949,-0.007176,-0.008967,-0.008446,0.027847,0.021186,0.065193,0.051932,0.039788,-0.001325,-0.012646
sdoh_DV,-0.017019,0.039155,-0.002313,0.122822,0.066374,0.072739,0.056812,nan,1.000000,0.140805,0.106416,-0.045242,0.047098,-0.014266,0.103076,0.041275,nan,nan,nan,-0.012161
sdoh_HIV,0.113594,0.148029,-0.014267,0.006955,-0.016115,0.007810,0.089178,-0.005949,0.140805,1.000000,0.116465,0.089663,0.024844,0.083351,-0.034997,0.074521,-0.015751,-0.010497,-0.001714,0.032330


In [ ]:
import pandas as pd

# Display options
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('expand_frame_repr', False)

# Function to check which patients have resource notes
def find_notes_left_behind_with_columns(notes_data, categorical_cols):
    results = []

    for record_id, group in notes_data.groupby('record_id'):
        patient_results = {
            'record_id': record_id,
            'resources notes left behind ': 0,
            'columns_with_notes': []
        }

        # Check each categorical column for any '1' values
        for col in categorical_cols:
            if group[col].infer_objects(copy=False).fillna(0).astype(bool).any():
                patient_results['resources notes left behind '] = 1
                patient_results['columns_with_notes'].append(col)

        results.append(patient_results)

    return pd.DataFrame(results)

# Assuming your input DataFrame is called 'notes_data' and your categorical variables list is 'Categoricals'
notes_left_behind_df = find_notes_left_behind_with_columns(notes_data, Categoricals)

# Count patients who have at least one resource note
if "resources notes left behind " in notes_left_behind_df.columns:
    filtered_df = notes_left_behind_df[notes_left_behind_df['resources notes left behind '] != 0]
else:
    filtered_df = notes_left_behind_df

# Number of patients who have resource notes left behind
patients_with_notes = filtered_df['record_id'].nunique()
print(f"Total patients having resource notes left behind: {patients_with_notes}")

# Total number of unique patients (regardless of notes)
total_patients = notes_data['record_id'].nunique()
print(f"Total number of patients in the dataset: {total_patients}")

# Patients without any notes
patients_without_notes = total_patients - patients_with_notes
print(f"Total patients with no resource notes: {patients_without_notes}")

# Optional: display the filtered DataFrame (patients with notes)
if "resources notes left behind " in filtered_df.columns:
    filtered_df = filtered_df.drop(columns=["resources notes left behind "])

print("\nFiltered list of patients with resource notes:")
print(filtered_df.to_string(index=False))


Total patients having resource notes left behind: 441
Total number of patients in the dataset: 2177
Total patients with no resource notes: 1736

Filtered list of patients with resource notes:
 record_id                                                                                  columns_with_notes
         9                                                                                     [emotional_res]
        11                                                                                     [emotional_res]
        12                                                                                         [other_res]
        21                                                                                          [food_res]
        26                                                                              [bill_rx_healthed_res]
        29                                                                                      [asthma_nores]
        31                     

In [ ]:
def calculate_resource_frequency(df):

    # Flatten the lists of resources
    all_resources = []
    for resources in df['columns_with_notes']:
        all_resources.extend(resources)

    # Create frequency table using pandas value_counts
    resource_counts = pd.Series(all_resources).value_counts().reset_index()
    resource_counts.columns = ['Resource', 'Frequency']

    # Calculate percentages
    total_notes = len(all_resources)
    resource_counts['Percentage'] = (resource_counts['Frequency'] / total_notes * 100).round(2)

    return resource_counts

# Use the function on your filtered DataFrame
resource_frequency_table = calculate_resource_frequency(filtered_df)

# Display the results
print("\nResource Notes Frequency Table:")
print(resource_frequency_table.to_string(index=False))


Resource Notes Frequency Table:
              Resource  Frequency  Percentage
             other_res        134       16.81
  bill_rx_healthed_res        100       12.55
             trans_res         92       11.54
              food_res         91       11.42
              util_res         83       10.41
               PCP_res         37        4.64
         emotional_res         30        3.76
        home_equip_res         27        3.39
            employ_res         24        3.01
     alc_substance_res         18        2.26
             covid_res         17        2.13
               hiv_res         15        1.88
          diabetes_res         15        1.88
         insurance_res         15        1.88
           housing_res         15        1.88
       insurance_nores         13        1.63
            asthma_res         10        1.25
          employ_nores          8        1.00
 cloth_child_phone_res          6        0.75
   alc_substance_nores          5        0.63
 

In [ ]:
# Function to calculate readmissions for each resource, including patients without any notes
def calculate_readmissions_by_resource(df, notes_data):
    # Get frequency counts first
    all_resources = []
    for resources in df['columns_with_notes']:
        all_resources.extend(resources)

    resource_counts = pd.Series(all_resources).value_counts().reset_index()
    resource_counts.columns = ['Resource', 'Frequency']

    # Calculate readmissions for each resource
    readmission_counts = []

    for resource in resource_counts['Resource']:
        # Get record_ids
        records_with_resource = df[df['columns_with_notes'].apply(lambda x: resource in x)]['record_id']

        # Count readmissions for records
        total_readmits = notes_data[
            (notes_data['record_id'].isin(records_with_resource)) & (notes_data['day_readmit'] == 1)
        ]['record_id'].nunique()

        readmission_counts.append(total_readmits)

    # Add readmission counts to the DataFrame
    resource_counts['Total Readmitted'] = readmission_counts

    # Calculate percentages
    resource_counts['Readmission Rate'] = (
        resource_counts['Total Readmitted'] / resource_counts['Frequency'] * 100
    ).round(2)

    # Sort by total readmissions in descending order
    resource_counts = resource_counts.sort_values('Total Readmitted', ascending=False)

    # Now, add data for patients who do not have any resource notes left behind
    patients_with_notes = df['record_id'].unique()
    patients_without_notes = notes_data[~notes_data['record_id'].isin(patients_with_notes)]

    # Count total patients without notes
    total_patients_without_notes = patients_without_notes['record_id'].nunique()

    # Count readmitted patients among those without notes
    total_readmitted_without_notes = patients_without_notes[patients_without_notes['day_readmit'] == 1]['record_id'].nunique()

    # Calculate readmission rate
    readmission_rate_without_notes = round((total_readmitted_without_notes / total_patients_without_notes * 100), 2) if total_patients_without_notes != 0 else 0

    # Append this information as a new row
    resource_counts = pd.concat([resource_counts, pd.DataFrame([{
        'Resource': 'No Resource Notes',
        'Frequency': total_patients_without_notes,
        'Total Readmitted': total_readmitted_without_notes,
        'Readmission Rate': readmission_rate_without_notes
    }])], ignore_index=True)

    return resource_counts

# Compute readmission analysis including patients without notes
readmission_analysis = calculate_readmissions_by_resource(filtered_df, notes_data)

# Display results
print("\nResource Notes and Readmission Analysis:")
print(readmission_analysis.to_string(index=False))



Resource Notes and Readmission Analysis:
              Resource  Frequency  Total Readmitted  Readmission Rate
             other_res        134                42             31.34
  bill_rx_healthed_res        100                25             25.00
             trans_res         92                24             26.09
              food_res         91                21             23.08
              util_res         83                17             20.48
               PCP_res         37                10             27.03
         emotional_res         30                10             33.33
             covid_res         17                 7             41.18
        home_equip_res         27                 5             18.52
     alc_substance_res         18                 5             27.78
            asthma_res         10                 4             40.00
           housing_res         15                 4             26.67
          employ_nores          8               

In [ ]:
import pandas as pd

# Function to compute Yule's Q using the pre-computed readmission counts for "food_res" and "food_nores"
def calculate_yule_q_for_food_resources(resource_counts):
    # Extract values for "food_res"
    food_res_data = resource_counts[resource_counts['Resource'] == 'food_res']
    if food_res_data.empty:
        print("No data found for 'food_res'")
        return None
    a = int(food_res_data['Total Readmitted'].values[0])
    freq_food_res = int(food_res_data['Frequency'].values[0])
    b = freq_food_res - a

    # Extract values for "food_nores"
    food_nores_data = resource_counts[resource_counts['Resource'] == 'food_nores']
    if food_nores_data.empty:
        print("No data found for 'food_nores'")
        return None
    c = int(food_nores_data['Total Readmitted'].values[0])
    freq_food_nores = int(food_nores_data['Frequency'].values[0])
    d = freq_food_nores - c

    # Calculate Yule's Q: (a*d - b*c) / (a*d + b*c)
    denominator = (a * d + b * c)
    Q = (a * d - b * c) / denominator if denominator != 0 else None

    # Display the contingency table and Yule's Q
    print("\nContingency Table:")
    print("                     Readmitted    Not Readmitted")
    print(f"Food Resource         {a}              {b}")
    print(f"No Food Resource      {c}              {d}")
    print("\nYule's Q:", Q)

    return Q

# Compute and print Yule's Q for the food resource comparison
Q_value = calculate_yule_q_for_food_resources(readmission_analysis)



Contingency Table:
                     Readmitted    Not Readmitted
Food Resource         21              70
No Food Resource      3              0

Yule's Q: -1.0


In [ ]:
import pandas as pd

# Function to compute Yule's Q using the pre-computed readmission counts for "food_res" and "food_nores"
def calculate_yule_q_for_transportation_resources(resource_counts):
    # Extract values for "food_res"
    trans_res_data = resource_counts[resource_counts['Resource'] == 'trans_res']
    if trans_res_data.empty:
        print("No data found for 'trans_res'")
        return None
    a = int(trans_res_data['Total Readmitted'].values[0])
    freq_trans_res = int(trans_res_data['Frequency'].values[0])
    b = freq_trans_res - a

    # Extract values for "food_nores"
    trans_nores_data = resource_counts[resource_counts['Resource'] == 'trans_nores']
    if trans_nores_data.empty:
        print("No data found for 'trans_nores'")
        return None
    c = int(trans_nores_data['Total Readmitted'].values[0])
    freq_trans_nores = int(trans_nores_data['Frequency'].values[0])
    d = freq_trans_nores - c

    # Calculate Yule's Q: (a*d - b*c) / (a*d + b*c)
    denominator = (a * d + b * c)
    Q = (a * d - b * c) / denominator if denominator != 0 else None

    # Display the contingency table and Yule's Q
    print("\nContingency Table:")
    print("                     Readmitted    Not Readmitted")
    print(f"Transportation Resource         {a}              {b}")
    print(f"No Transportation Resource      {c}              {d}")
    print("\nYule's Q:", Q)

    return Q

# Compute and print Yule's Q for the food resource comparison
Q_value = calculate_yule_q_for_transportation_resources(readmission_analysis)



Contingency Table:
                     Readmitted    Not Readmitted
Transportation Resource         24              68
No Transportation Resource      2              0

Yule's Q: -1.0


In [ ]:
import pandas as pd

# Function to compute Yule's Q using the pre-computed readmission counts for "food_res" and "food_nores"
def calculate_yule_q_for_asthma_resources(resource_counts):
    # Extract values for "food_res"
    asthma_res_data = resource_counts[resource_counts['Resource'] == 'asthma_res']
    if asthma_res_data.empty:
        print("No data found for 'asthma_res'")
        return None
    a = int(asthma_res_data['Total Readmitted'].values[0])
    freq_athma_res = int(asthma_res_data['Frequency'].values[0])
    b = freq_athma_res - a

    # Extract values for "ashtma_nores"
    asthma_nores_data = resource_counts[resource_counts['Resource'] == 'asthma_nores']
    if asthma_nores_data.empty:
        print("No data found for 'asthma_nores'")
        return None
    c = int(asthma_nores_data['Total Readmitted'].values[0])
    freq_asthma_nores = int(asthma_nores_data['Frequency'].values[0])
    d = freq_asthma_nores - c

    # Calculate Yule's Q: (a*d - b*c) / (a*d + b*c)
    denominator = (a * d + b * c)
    Q = (a * d - b * c) / denominator if denominator != 0 else None

    # contingency table
    print("\nContingency Table:")
    print("                     Readmitted    Not Readmitted")
    print(f"Asthma Resource         {a}              {b}")
    print(f"No Asthma Resource      {c}              {d}")
    print("\nYule's Q:", Q)

    return Q

Q_value = calculate_yule_q_for_asthma_resources(readmission_analysis)



Contingency Table:
                     Readmitted    Not Readmitted
Asthma Resource         4              6
No Asthma Resource      0              2

Yule's Q: 1.0


In [ ]:
import pandas as pd

# Function to compute Yule's Q using the pre-computed readmission counts for "food_res" and "food_nores"
def calculate_yule_q_for_pcp_resources(resource_counts):
    # Extract values for "food_res"
    pcp_res_data = resource_counts[resource_counts['Resource'] == 'PCP_res']
    if pcp_res_data.empty:
        print("No data found for 'PCP_res'")
        return None
    a = int(pcp_res_data['Total Readmitted'].values[0])
    freq_pcp_res = int(pcp_res_data['Frequency'].values[0])
    b = freq_pcp_res - a

    # Extract values for "ashtma_nores"
    pcp_nores_data = resource_counts[resource_counts['Resource'] == 'PCP_nores']
    if pcp_nores_data.empty:
        print("No data found for 'PCP_nores'")
        return None
    c = int(pcp_nores_data['Total Readmitted'].values[0])
    freq_pcp_nores = int(pcp_nores_data['Frequency'].values[0])
    d = freq_pcp_nores - c

    # Calculate Yule's Q: (a*d - b*c) / (a*d + b*c)
    denominator = (a * d + b * c)
    Q = (a * d - b * c) / denominator if denominator != 0 else None

    # contingency table
    print("\nContingency Table:")
    print("                     Readmitted    Not Readmitted")
    print(f"PCP Resource         {a}              {b}")
    print(f"No PCP Resource      {c}              {d}")
    print("\nYule's Q:", Q)

    return Q

Q_value = calculate_yule_q_for_pcp_resources(readmission_analysis)



Contingency Table:
                     Readmitted    Not Readmitted
PCP Resource         10              27
No PCP Resource      2              3

Yule's Q: -0.2857142857142857


In [ ]:
import pandas as pd

# Function to compute Yule's Q using the pre-computed readmission counts for "food_res" and "food_nores"
def calculate_yule_q_for_employ_resources(resource_counts):
    # Extract values for "food_res"
    employ_res_data = resource_counts[resource_counts['Resource'] == 'employ_res']
    if employ_res_data.empty:
        print("No data found for 'employ_res'")
        return None
    a = int(employ_res_data['Total Readmitted'].values[0])
    freq_employ_res = int(employ_res_data['Frequency'].values[0])
    b = freq_employ_res - a

    # Extract values for "ashtma_nores"
    employ_nores_data = resource_counts[resource_counts['Resource'] == 'employ_nores']
    if employ_nores_data.empty:
        print("No data found for 'employ_nores'")
        return None
    c = int(employ_nores_data['Total Readmitted'].values[0])
    freq_employ_nores = int(employ_nores_data['Frequency'].values[0])
    d = freq_employ_nores - c

    # Calculate Yule's Q: (a*d - b*c) / (a*d + b*c)
    denominator = (a * d + b * c)
    Q = (a * d - b * c) / denominator if denominator != 0 else None

    # contingency table
    print("\nContingency Table:")
    print("                     Readmitted    Not Readmitted")
    print(f"Employ Resource         {a}              {b}")
    print(f"No Employ Resource      {c}              {d}")
    print("\nYule's Q:", Q)

    return Q

Q_value = calculate_yule_q_for_employ_resources(readmission_analysis)



Contingency Table:
                     Readmitted    Not Readmitted
Employ Resource         3              21
No Employ Resource      3              5

Yule's Q: -0.6153846153846154
